# Concrete Strength Prediction

**Regression Project 58**

Full pipeline: EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.

## 2. Project Overview

Predict the compressive strength of concrete given its mix design (cement, water, aggregates,
superplasticizer, fly ash, slag) and curing age. The UCI Concrete Compressive Strength dataset
contains 1,030 samples with 8 input features and 1 target.

Concrete strength prediction is a core task in civil engineering — it determines structural
safety and is traditionally done via expensive and time-consuming physical testing.

## 3. Learning Objectives

1. Build a regression pipeline for materials science
2. Understand how concrete mix proportions affect strength
3. Learn the critical role of curing age in concrete development
4. Compare linear vs non-linear models on non-linear physical data
5. Evaluate with R², RMSE, MAE and residual analysis
6. Practice feature engineering with domain knowledge

## 4. Problem Statement

> Given the concrete mix design and curing age, predict the **compressive strength** (MPa).

This is a **regression** task. Primary metric: **R²**; secondary: RMSE, MAE.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Civil engineers | Structural safety verification |
| Construction companies | Mix optimization to reduce costs |
| Quality control | Reduce reliance on destructive testing |
| Sustainability | Optimize cement usage (major CO₂ source) |
| ML learners | Non-linear regression with material science context |

## 6. Dataset Overview

| Property | Value |
|---|---|
| Source | Kaggle: `elikplim/concrete-compressive-strength-data-set` |
| Records | 1,030 |
| Features | 8 (Cement, Blast Furnace Slag, Fly Ash, Water, Superplasticizer, Coarse Aggregate, Fine Aggregate, Age) |
| Target | Concrete Compressive Strength (MPa) |
| Key insight | Cement content and age are strongest predictors |

## 7. Dataset Source and License Notes

- **Kaggle**: [elikplim/concrete-compressive-strength-data-set](https://www.kaggle.com/datasets/elikplim/concrete-compressive-strength-data-set)
- Originally from UCI ML Repository (Yeh, 1998)
- License: CC0 / Public Domain
- Downloaded via `kagglehub` at runtime

## 8. Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ['lazypredict', 'flaml', 'kagglehub', 'xgboost', 'lightgbm']:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Environment ready.')

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42
print('All imports loaded.')

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = 'elikplim/concrete-compressive-strength-data-set'
TARGET = 'csMPa'  # Will be identified during loading
TEST_SIZE = 0.15
VAL_SIZE = 0.15
FLAML_BUDGET = 90
MAX_ROWS = 50000

## 11. Dataset Download or Loading

Download the Concrete Compressive Strength dataset via `kagglehub`.

In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f'Downloaded to: {path}')
except Exception as e:
    raise RuntimeError(
        f'Dataset download failed: {e}\n'
        'Ensure KAGGLE_API_TOKEN is set. '
        'Fallback: download from https://www.kaggle.com/datasets/elikplim/concrete-compressive-strength-data-set'
    )
all_files = glob.glob(os.path.join(path, '**', '*.*'), recursive=True)
csv_files = [f for f in all_files if f.endswith('.csv')]
xlsx_files = [f for f in all_files if f.endswith('.xlsx') or f.endswith('.xls')]
if csv_files:
    df_raw = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
elif xlsx_files:
    df_raw = pd.read_excel(sorted(xlsx_files, key=os.path.getsize, reverse=True)[0])
else:
    raise FileNotFoundError(f'No data files in {path}')
print(f'Shape: {df_raw.shape}')
print(f'Columns: {df_raw.columns.tolist()}')
df_raw.head()

## 12. Data Validation Checks

Identify the target column (compressive strength) and validate data quality.

In [ ]:
df = df_raw.copy()
# Identify target: look for 'strength' or 'csMPa' in column names
target_candidates = [c for c in df.columns if 'strength' in c.lower() or 'csmpa' in c.lower() or 'compressive' in c.lower()]
if target_candidates:
    TARGET = target_candidates[0]
else:
    TARGET = df.columns[-1]  # Last column is typically the target
print(f'Target column: {TARGET}')

# Shorten column names for convenience
name_map = {}
for c in df.columns:
    short = c.replace('Concrete compressive strength(MPa, megapascals) ', 'Strength_MPa')
    short = short.split('(')[0].strip().replace(' ', '_')
    name_map[c] = short
df = df.rename(columns=name_map)
TARGET = name_map.get(TARGET, TARGET)
print(f'Renamed target: {TARGET}')

print(f'\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
print(f'Duplicated rows: {df.duplicated().sum()}')
df = df.dropna(subset=[TARGET]).drop_duplicates().reset_index(drop=True)
print(f'Shape: {df.shape}')
df.dtypes

## 13. Exploratory Data Analysis

All 8 features are numeric — mix components (kg/m³) and age (days).

In [ ]:
df.describe()

In [ ]:
num_feats = [c for c in df.columns if c != TARGET]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, num_feats[:8]):
    df[col].hist(bins=25, ax=ax, alpha=0.7, color='steelblue')
    ax.set_title(col, fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout(); plt.show()

In [ ]:
# Top correlations with target
top_corr = df.corr(numeric_only=True)[TARGET].drop(TARGET).abs().sort_values(ascending=False)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, top_corr.head(4).index):
    ax.scatter(df[col], df[TARGET], alpha=0.3, s=10)
    ax.set_xlabel(col); ax.set_ylabel(TARGET)
    ax.set_title(f'{col} vs Strength')
plt.tight_layout(); plt.show()

## 14. Target Analysis

Concrete strength ranges from ~2 to ~82 MPa, with a roughly normal distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET], bins=40, color='coral', alpha=0.7)
axes[0].set_title('Compressive Strength Distribution (MPa)')
axes[1].boxplot(df[TARGET].dropna(), vert=True)
axes[1].set_title('Strength Box Plot')
plt.tight_layout(); plt.show()
print(f'Mean: {df[TARGET].mean():.1f} MPa')
print(f'Median: {df[TARGET].median():.1f} MPa')
print(f'Std: {df[TARGET].std():.1f} MPa')
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15. Train / Validation / Test Split

70/15/15 split with fixed seed.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## 16. Preprocessing Strategy

- All features numeric — impute median + StandardScaler
- Some mix components (fly ash, slag) have many zeros — valid, not missing

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols)
], remainder='drop')
print(f'Numeric features: {len(num_cols)}')

## 17. Feature Engineering

- **Water-cement ratio**: Classic indicator of concrete strength
- **Total binder**: Cement + Slag + Fly Ash
- **Log age**: Logarithm of curing age (strength grows logarithmically with time)

In [ ]:
def engineer_features(d):
    d = d.copy()
    cols_lower = {c.lower(): c for c in d.columns}
    cement_col = next((cols_lower[k] for k in cols_lower if 'cement' in k), None)
    water_col = next((cols_lower[k] for k in cols_lower if 'water' in k), None)
    slag_col = next((cols_lower[k] for k in cols_lower if 'slag' in k or 'blast' in k), None)
    ash_col = next((cols_lower[k] for k in cols_lower if 'ash' in k or 'fly' in k), None)
    age_col = next((cols_lower[k] for k in cols_lower if 'age' in k), None)
    if cement_col and water_col:
        d['water_cement_ratio'] = d[water_col] / d[cement_col].replace(0, np.nan)
        d['water_cement_ratio'] = d['water_cement_ratio'].fillna(0)
    if cement_col:
        binder = d[cement_col].copy()
        if slag_col: binder = binder + d[slag_col]
        if ash_col: binder = binder + d[ash_col]
        d['total_binder'] = binder
    if age_col:
        d['log_age'] = np.log1p(d[age_col])
    return d

X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols)
], remainder='drop')
print(f'Features after engineering: {X_train.shape[1]}')

## 18. Baseline Model

In [ ]:
results = {}
for name, reg in [
    ('Dummy (mean)', DummyRegressor(strategy='mean')),
    ('LinearRegression', LinearRegression()),
    ('RandomForest', RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1))
]:
    pipe = Pipeline([('pre', preprocessor), ('reg', reg)])
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_val)
    r = {'RMSE': np.sqrt(mean_squared_error(y_val, yp)),
         'MAE': mean_absolute_error(y_val, yp),
         'R2': r2_score(y_val, yp)}
    results[name] = r
    print(f"{name}: R2={r['R2']:.4f}, RMSE={r['RMSE']:.2f} MPa, MAE={r['MAE']:.2f} MPa")

## 19. LazyPredict Benchmark

In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)

## 20. FLAML AutoML Run

In [ ]:
automl = AutoML()
automl.fit(X_train_lp, y_train, task='regression', metric='r2',
           time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f'Best estimator: {automl.best_estimator}')
yp_fl = automl.predict(X_val_lp)
results['FLAML'] = {'RMSE': np.sqrt(mean_squared_error(y_val, yp_fl)),
                    'MAE': mean_absolute_error(y_val, yp_fl),
                    'R2': r2_score(y_val, yp_fl)}
print(f"FLAML: R2={results['FLAML']['R2']:.4f}, RMSE={results['FLAML']['RMSE']:.2f} MPa")

## 21. Top 3 Model Selection

In [ ]:
try:
    from lightgbm import LGBMRegressor; has_lgbm = True
except ImportError: has_lgbm = False
try:
    from xgboost import XGBRegressor; has_xgb = True
except ImportError: has_xgb = False
top3 = {}
if has_lgbm:
    top3['LightGBM'] = LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesRegressor
    top3['ExtraTrees'] = ExtraTreesRegressor(n_estimators=500, random_state=SEED, n_jobs=-1)
if has_xgb:
    top3['XGBoost'] = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, n_jobs=-1)
else:
    from sklearn.ensemble import AdaBoostRegressor
    top3['AdaBoost'] = AdaBoostRegressor(n_estimators=300, random_state=SEED)
top3['GradBoosting'] = GradientBoostingRegressor(n_estimators=400, learning_rate=0.05, max_depth=5, random_state=SEED)
print(f'Top 3: {list(top3.keys())}')

## 22. Final Training and Evaluation of Top 3

In [ ]:
X_tr_t = preprocessor.fit_transform(X_train)
X_te_t = preprocessor.transform(X_test)
final = {}
predictions = {}
for name, mdl in top3.items():
    mdl.fit(X_tr_t, y_train)
    yp = mdl.predict(X_te_t)
    predictions[name] = yp
    final[name] = {'RMSE': np.sqrt(mean_squared_error(y_test, yp)),
                   'MAE': mean_absolute_error(y_test, yp),
                   'R2': r2_score(y_test, yp)}
    print(f"{name}: R2={final[name]['R2']:.4f}, RMSE={final[name]['RMSE']:.2f} MPa")
pd.DataFrame(final).T.sort_values('R2', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    ax.scatter(y_test, yp, alpha=0.5, s=15)
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=2)
    ax.set_xlabel('Actual (MPa)'); ax.set_ylabel('Predicted (MPa)')
    ax.set_title(f"{name} (R2={final[name]['R2']:.3f})")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    residuals = y_test.values - yp
    ax.scatter(yp, residuals, alpha=0.5, s=15)
    ax.axhline(0, color='red', linestyle='--')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Residual')
    ax.set_title(f'{name} Residuals')
plt.tight_layout(); plt.show()

## 23. Error Analysis

In [ ]:
best_name = max(final, key=lambda k: final[k]['R2'])
best_preds = predictions[best_name]
abs_errors = np.abs(y_test.values - best_preds)
print(f'Best model: {best_name}')
print(f'Mean absolute error: {abs_errors.mean():.2f} MPa')
print(f'Median absolute error: {np.median(abs_errors):.2f} MPa')
print(f'90th percentile error: {np.percentile(abs_errors, 90):.2f} MPa')
print(f'Max error: {abs_errors.max():.2f} MPa')

## 24. Interpretation and Business Insight

- **Cement content** is the dominant strength driver — more cement = stronger concrete
- **Age** (curing time) is the second strongest — strength increases logarithmically with time
- **Water-cement ratio** is inversely related — classic Abrams' law in concrete technology
- **Superplasticizer** enables higher strength at lower water content
- **Fly ash and slag** (supplementary cementitious materials) contribute to long-term strength
- Non-linear models are essential because water-cement interaction is inherently non-linear

## 25. Limitations

1. Only 1,030 samples — limited coverage of mix space
2. No aggregate type/source information
3. No environmental curing conditions (temperature, humidity)
4. No admixture details beyond superplasticizer dosage
5. Compressive strength only — no tensile/flexural strength

## 26. How to Improve This Project

1. Include aggregate properties and curing conditions
2. Log-transform age feature (strength ∝ log(age))
3. Multi-output: predict strength at multiple ages simultaneously
4. Physics-informed features (w/c ratio bins, binder intensity)
5. Cross-validation for robust estimates on 1K samples

## 27. Production Considerations

- Mix design optimization tool for concrete plants
- Quality prediction before pouring (save 28-day waiting)
- Cost optimization (minimize cement while meeting strength spec)
- Carbon footprint reduction through optimized mixes
- Real-time strength monitoring integration

## 28. Common Mistakes

1. Not creating water-cement ratio feature
2. Treating age as linear (it's logarithmic)
3. Not recognizing that zeros in fly ash/slag are valid, not missing
4. Overfitting on 1,030 rows with complex deep learning
5. Ignoring domain knowledge from concrete science

## 29. Mini Challenge / Exercises

1. Add Abrams' law feature: strength ∝ 1/w_c_ratio
2. Predict strength at 28 days only (filter by age == 28)
3. Try SVR with RBF kernel vs gradient boosting
4. Use SHAP to visualize feature importance
5. Build a mix optimization tool that maximizes predicted strength for a cost budget

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Regression — predict concrete compressive strength |
| Dataset | UCI Concrete (1,030 records) |
| Difficulty | Medium |
| Key insight | Cement and age dominate; w/c ratio is critical |
| Best approach | Gradient boosting with engineered ratios |
| Primary metric | R² |
| Baseline comparison | Non-linear models far outperform linear regression |